# 6. From a question to a reproducible student project

A simulation project is more than a notebook that produces an
attractive image. It connects a biological question to a measurable
prediction, records the model and software version, tests invariants
and separates generated results from source files.

**Learning objectives**

- formulate a measurable simulation question;
- construct and save a complete ModelSpec;
- add a small registered interaction without editing BioLGCA internals;
- test the interaction's conservation claim; and
- organize files so another person can reproduce the run.


In [ ]:
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np

from lgca.model import (
    AnalysisSpec,
    Description,
    ModelSpec,
    SpaceSpec,
    StateSpec,
    TimeSpec,
    run_model,
    save_model_spec,
)
from lgca.pipeline import InteractionPipelineSpec
from lgca.plugins import (
    ConservationLaw,
    PluginInfo,
    ReorientationOperator,
    describe_plugin,
    register_plugin,
)
from lgca.simulation import DensityRecorder, NodeRecorder, PopulationRecorder

print("BioLGCA version:", version("biolgca"))


## Question, prediction and observable

**Question:** Does a deterministic clockwise channel rotation create
a reproducible circulation bias relative to unbiased random walk?

**Prediction:** repeated rotation changes the direction of flux but
does not change total particle number.

**Observables:** total population checks the conservation law, while
the summed flux vector measures directional change. The conservation
check should become an automated unit test when the interaction is
moved from exploration into a reusable project module.


## Define a small interaction plugin

The operator below rotates the velocity channels at every interior
node. It changes direction but neither creates nor removes occupied
channels. The metadata tells the pipeline which phase it belongs to
and what it claims to conserve.

This is project code, not a copy of a BioLGCA implementation. In a
real project, place it in `interactions.py`, import that module before
loading a model file, and test it independently.


In [ ]:
class RotateVelocityChannelsOperator(ReorientationOperator):
    def __init__(self, info, parameters=None):
        super().__init__(info=info, parameters=parameters)

    def validate(self, context):
        if context.spec.state.identity_based:
            raise ValueError("tutorial rotation supports classical states only")
        if not context.spec.state.volume_exclusion:
            raise ValueError("tutorial rotation requires volume exclusion")
        if context.spec.state.n_species != 1:
            raise ValueError("tutorial rotation supports one species")

    def apply(self, context, step):
        lgca = context.lgca
        interior = lgca.nodes[lgca.nonborder]
        velocity_state = interior[..., : lgca.velocitychannels].copy()
        interior[..., : lgca.velocitychannels] = np.roll(
            velocity_state,
            shift=1,
            axis=-1,
        )


ROTATE_INFO = PluginInfo(
    name="tutorial.rotate_velocity_channels",
    operator_kind="reorientation",
    backend_families=("classical",),
    parameters={},
    conservation_law=ConservationLaw(
        conserves_total_particles=True,
        conserves_phenotype_particles=True,
        conserves_momentum=False,
        changes=("channel direction",),
    ),
    port_status="tutorial",
    test_status="notebook_tested",
    description="Rotate velocity channels without changing occupancy.",
)


def rotate_factory(parameters=None):
    return RotateVelocityChannelsOperator(ROTATE_INFO, parameters)


try:
    describe_plugin(ROTATE_INFO.name)
except KeyError:
    register_plugin(ROTATE_INFO, rotate_factory)


## Add the interaction to a visible specification

Registration gives the interaction a stable name. The ModelSpec now
refers to that name exactly as it would refer to a built-in plugin.
Propagation is disabled for the one-step invariant test so that the
interaction is isolated.


In [ ]:
project_spec = ModelSpec(
    description=Description(
        title="Clockwise channel rotation",
        details="Student project model used to test a conserving reorientation.",
        tags=("student-project", "custom-interaction"),
    ),
    space=SpaceSpec(
        geometry="square",
        dims=(10, 10),
        boundary="periodic",
    ),
    state=StateSpec(
        density=0.2,
        restchannels=0,
    ),
    time=TimeSpec(
        steps=1,
        seed=61,
    ),
    dynamics=InteractionPipelineSpec(
        operators=[{"name": "tutorial.rotate_velocity_channels"}],
        propagation=False,
    ),
    analysis=AnalysisSpec(
        observers=[NodeRecorder(), DensityRecorder(), PopulationRecorder()],
    ),
)

project_result = run_model(project_spec, showprogress=False)
before = project_result.lgca.nodes_t[0]
after = project_result.lgca.nodes_t[1]

assert before.sum() == after.sum()
assert project_result.lgca.n_t[0] == project_result.lgca.n_t[-1]
print("conserved population:", project_result.lgca.n_t.tolist())


In [ ]:
before_flux = project_result.lgca.calc_flux(before).sum(axis=(0, 1))
after_flux = project_result.lgca.calc_flux(after).sum(axis=(0, 1))
print("flux before:", before_flux)
print("flux after: ", after_flux)

fig, axes = plt.subplots(1, 2, figsize=(7, 3), constrained_layout=True)
axes[0].imshow(before.sum(axis=-1).T, origin="lower")
axes[0].set_title("before interaction")
axes[1].imshow(after.sum(axis=-1).T, origin="lower")
axes[1].set_title("after interaction")
plt.show()
plt.close(fig)


The density maps are identical because channel rotation changes
direction, not node occupancy. The flux vectors reveal the changed
channel state. This illustrates why an invariant and a question-
specific observable test different aspects of an interaction.


## Save the model and provenance

A shareable run records the model specification, random seed and
package version. A custom plugin's Python module must accompany the
model file; importing trusted plugin code is deliberately separate
from parsing model data.


In [ ]:
with TemporaryDirectory() as temporary_directory:
    run_directory = Path(temporary_directory)
    model_path = save_model_spec(project_spec, run_directory / "model.json")
    metadata = {
        "biolgca_version": version("biolgca"),
        "seed": project_spec.time.seed,
        "model_file": model_path.name,
        "interaction_module": "interactions.py",
    }
    print(model_path.read_text(encoding="utf-8")[:400] + "...")
    print(metadata)


## Suggested project layout

```text
my-lgca-project/
|-- README.md                 # question, prediction and run commands
|-- model.json                # saved ModelSpec
|-- interactions.py           # registered custom interaction
|-- analysis.py               # reusable observables
|-- notebooks/
|   `-- exploration.ipynb     # exploratory narrative, no hidden state
|-- tests/
|   `-- test_interactions.py  # conservation and compatibility tests
`-- results/                  # generated files, normally ignored by git
```

Record the BioLGCA version and seeds with every result. Keep raw
simulation output separate from analysis figures so results can be
regenerated without editing source files.

## Project checklist

1. State a biological question and measurable prediction.
2. Start by composing built-in terms and phases visibly.
3. Add custom code only when the mechanism is genuinely missing.
4. Test conservation laws and supported model families.
5. Choose observables before inspecting the most favorable run.
6. Use multiple prespecified seeds and report variability.
7. Save ModelSpec, package version, custom plugin module and analysis code.

## Exercises

1. Re-enable propagation and predict the combined effect of rotation
   and movement.
2. Add a parameter controlling clockwise versus counter-clockwise rotation.
3. Move the operator into a module and write a pytest invariant for
   empty, partially occupied and fully occupied channel states.
